# 14.8 — Time-series anomaly detection

A time-series anomaly is a value or pattern that is unlikely after accounting for local trend, seasonality, and expected noise.

Anomaly detection is residual modeling: first estimate what should have happened, then standardize the surprise. Robust scale estimates keep the anomaly itself from hiding inside the threshold. Save a copy to Drive to edit.

In [ ]:

import numpy as np
import matplotlib.pyplot as plt

SEED = 1306
rng = np.random.default_rng(SEED)


def rmse(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    return float(np.sqrt(np.mean((y_true - y_pred) ** 2)))


def interval_coverage(y_true, lower, upper):
    y_true = np.asarray(y_true, dtype=float)
    lower = np.asarray(lower, dtype=float)
    upper = np.asarray(upper, dtype=float)
    inside = (y_true >= lower) & (y_true <= upper)
    return float(np.mean(inside))


def f13_time_series_ladder():
    t = np.arange(72, dtype=float)
    rungs = []

    y1 = np.full_like(t, 10.0)
    rungs.append({"name": "D1 constant", "t": t, "y": y1, "true": y1, "period": 12})

    y2 = 8.0 + 0.18 * t
    rungs.append({"name": "D2 drift", "t": t, "y": y2, "true": y2, "period": 12})

    noise3 = rng.normal(0.0, 0.45, size=t.size)
    true3 = 8.0 + 0.18 * t
    y3 = true3 + noise3
    rungs.append({"name": "D3 drift + noise", "t": t, "y": y3, "true": true3, "period": 12})

    seasonal4 = 1.8 * np.sin(2.0 * np.pi * t / 12.0)
    true4 = 8.0 + 0.12 * t + seasonal4
    y4 = true4 + rng.normal(0.0, 0.35, size=t.size)
    rungs.append({"name": "D4 seasonal", "t": t, "y": y4, "true": true4, "period": 12})

    seasonal5 = 1.8 * np.sin(2.0 * np.pi * t / 12.0)
    true5 = 8.0 + 0.10 * t + seasonal5
    true5 = true5 + np.where(t >= 48.0, 3.0 + 0.06 * (t - 48.0), 0.0)
    y5 = true5 + rng.normal(0.0, 0.45, size=t.size)
    y5[18] = y5[18] + 5.0
    y5[55] = y5[55] - 4.0
    rungs.append({"name": "D5 outliers + regime shift", "t": t, "y": y5, "true": true5, "period": 12})

    return rungs


def train_test_split_series(rung, train_size=48):
    train = {"t": rung["t"][:train_size], "y": rung["y"][:train_size]}
    test = {"t": rung["t"][train_size:], "y": rung["y"][train_size:], "true": rung["true"][train_size:]}
    return train, test


def print_ladder_preview(rungs):
    for rung in rungs:
        name = rung["name"]
        y = rung["y"]
        sample = np.round(y[:6], 3)
        print(f"{name}: n={y.size}, period={rung['period']}, first six={sample}")


def plot_forecast_panels(results, metric_name="RMSE", marker_key=None, component_key=None):
    fig, axes = plt.subplots(2, 3, figsize=(14, 7))
    axes = axes.ravel()

    for idx, result in enumerate(results):
        ax = axes[idx]
        ax.plot(result["test_t"], result["test_y"], label="observed", color="black")
        ax.plot(result["test_t"], result["forecast"], label="forecast", color="tab:blue")
        ax.plot(result["test_t"], result["test_true"], label="true signal", color="tab:green", alpha=0.7)
        if component_key is not None and component_key in result:
            ax.plot(result["test_t"], result[component_key], label=component_key, color="tab:orange", alpha=0.8)
        if marker_key is not None and marker_key in result:
            marks = result[marker_key]
            for mark in np.atleast_1d(marks):
                ax.axvline(mark, color="tab:red", linestyle="--", alpha=0.6)
        ax.set_title(f"{result['name']}\n{metric_name}={result['metric']:.3f}")
        ax.grid(True, alpha=0.25)

    axes[-1].plot(range(1, len(results) + 1), [r["metric"] for r in results], marker="o")
    axes[-1].set_xticks(range(1, len(results) + 1))
    axes[-1].set_xlabel("D-rung")
    axes[-1].set_ylabel(metric_name)
    axes[-1].set_title(f"{metric_name} curve")
    axes[-1].grid(True, alpha=0.25)
    axes[0].legend(loc="best", fontsize=8)
    fig.tight_layout()



## The concept, built once (D1)

An anomaly score standardizes the residual:

$$z_t=\frac{y_t-\hat y_t}{\hat\sigma_t}$$

The robust alternative replaces mean and standard deviation with median and MAD. The lesson checks are $28.284271$, $26.979630$, and a seasonal residual of exactly $5.000$.


In [ ]:

def residual_anomaly_score(baseline, candidate, robust=False):
    baseline = np.asarray(baseline, dtype=float)
    candidate = float(candidate)
    if robust:
        center = float(np.median(baseline))
        mad = float(np.median(np.abs(baseline - center)))
        scale = 1.4826 * mad
    else:
        center = float(np.mean(baseline))
        scale = float(np.std(baseline))
    return (candidate - center) / scale

standard_score = residual_anomaly_score([10.0, 11.0, 9.0, 10.0], 30.0, robust=False)
robust_score = residual_anomaly_score([10.0, 11.0, 9.0, 10.0], 30.0, robust=True)
t = np.array([3.0])
expected = 10.0 + 2.0 * np.sin(2.0 * np.pi * t / 12.0)
observed = expected + 5.0
seasonal_residual = float(observed[0] - expected[0])

assert np.isclose(standard_score, 28.284271, atol=5e-7)
assert np.isclose(robust_score, 26.979630, atol=5e-7)
assert np.isclose(seasonal_residual, 5.000, atol=1e-12)
print("lesson checks passed")



For a full series, estimate a trend-plus-season baseline on the training window, score held-out residuals, and convert thresholds into coverage. Robust scale is estimated from held-out-clean training residuals.


In [ ]:

def fit_seasonal_baseline(train_t, train_y, period=12):
    train_t = np.asarray(train_t, dtype=float)
    train_y = np.asarray(train_y, dtype=float)
    columns = [np.ones_like(train_t), train_t]
    for k in range(1, 3):
        angle = 2.0 * np.pi * k * train_t / period
        columns.append(np.sin(angle))
        columns.append(np.cos(angle))
    X = np.column_stack(columns)
    beta = np.linalg.lstsq(X, train_y, rcond=None)[0]

    def predict(new_t):
        new_t = np.asarray(new_t, dtype=float)
        cols = [np.ones_like(new_t), new_t]
        for k in range(1, 3):
            angle = 2.0 * np.pi * k * new_t / period
            cols.append(np.sin(angle))
            cols.append(np.cos(angle))
        return np.column_stack(cols) @ beta

    fitted = predict(train_t)
    residuals = train_y - fitted
    center = float(np.median(residuals))
    mad = float(np.median(np.abs(residuals - center)))
    robust_scale = max(1.4826 * mad, 1e-6)
    return {"predict": predict, "scale": robust_scale, "center": center, "train_residuals": residuals}


def flag_anomalies(y, forecast, center, scale, threshold=3.5):
    residual = np.asarray(y, dtype=float) - np.asarray(forecast, dtype=float)
    score = np.abs((residual - center) / scale)
    return score, score > threshold



## The dataset ladder

We use the same F13 ladder in every notebook so the method faces increasing time-series difficulty inline: D1 constant, D2 drift, D3 drift plus noise, D4 monthly seasonality, and D5 outliers plus a real regime shift.


In [ ]:

rungs = f13_time_series_ladder()
print_ladder_preview(rungs)


In [ ]:

results = []

for rung in rungs:
    train, test = train_test_split_series(rung)
    model = fit_seasonal_baseline(train["t"], train["y"], period=rung["period"])
    forecast = model["predict"](test["t"])
    scores, flags = flag_anomalies(test["y"], forecast, model["center"], model["scale"])
    lower = forecast + model["center"] - 3.5 * model["scale"]
    upper = forecast + model["center"] + 3.5 * model["scale"]
    metric = rmse(test["true"], forecast)
    cov = interval_coverage(test["y"], lower, upper)
    results.append({
        "name": rung["name"],
        "test_t": test["t"],
        "test_y": test["y"],
        "test_true": test["true"],
        "forecast": forecast,
        "metric": metric,
        "coverage": cov,
        "anomaly_times": test["t"][flags],
    })

for result in results:
    print(f"{result['name']}: residual RMSE={result['metric']:.3f}, coverage={result['coverage']:.3f}, flags={len(result['anomaly_times'])}")


In [ ]:

plot_forecast_panels(results, metric_name="RMSE", marker_key="anomaly_times")



## Pitfall on D5: letting anomalies train the threshold

A contaminated standard deviation can inflate after spikes and make later anomalies look normal. The fix is a robust or held-out threshold based on median absolute deviation.


In [ ]:

d5 = rungs[-1]
contaminated = d5["y"][:56]
recent_center = float(np.mean(contaminated))
recent_scale = float(np.std(contaminated))
robust_center = float(np.median(contaminated))
robust_mad = float(np.median(np.abs(contaminated - robust_center)))
robust_scale = 1.4826 * robust_mad
candidate = float(d5["y"][55])
contaminated_score = abs((candidate - recent_center) / recent_scale)
robust_score = abs((candidate - robust_center) / robust_scale)
print(f"contaminated z={contaminated_score:.3f}")
print(f"robust z={robust_score:.3f}")
print("The robust score keeps the D5 shock visible instead of letting it set its own threshold.")



## Evaluate it + Practice

- Track residual RMSE and threshold coverage; anomalies should be visible without making every seasonal peak an alert.
- Sanity check: score residuals, not raw values, especially on D4 and D5.
- Ablation: use ordinary standard deviation after injecting outliers; robust MAD thresholds should be more stable.
- Failure signal: the estimated scale grows after an anomaly and hides later surprises.
- Compare spike flags with sustained residual signs before calling something a changepoint.

Practice prompts:
1. Lower the robust threshold from 3.5 to 2.5 and count false alarms on D4.


2. Modify the hardest rung and rerun the same metric table.

3. Write one sentence explaining whether RMSE or coverage is the better decision metric here.